In [0]:
"""
02_quality_kpis.py

Manufacturing Quality KPIs

Source:
    fact_quality

Target:
    quality_kpis

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    sum,
    when,
)

# ============================================================
# Quality KPIs
# ============================================================

@dlt.table(
    name="quality_kpis",
    comment="Manufacturing quality KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def quality_kpis():

    quality = dlt.read("fact_quality")

    return (

        quality

        .groupBy(

            "plant_code",
            "product_code",
            "product_name",
            "family",

        )

        .agg(

            count("*").alias(
                "tests_completed"
            ),

            sum(

                when(

                    col("result") == "PASS",

                    1

                ).otherwise(0)

            ).alias(
                "passed_tests"
            ),

         (
            count("*")
            - sum(
                when(
                    col("result") == "PASS",
                    1
                ).otherwise(0)
            )
        ).alias("failed_tests"),

            avg(

                "target_value"

            ).alias(
                "average_target_value"
            ),

            avg(

                "measured_value"

            ).alias(
                "average_measured_value"
            ),

        )

        .withColumn(

            "pass_rate",

            when(

                col("tests_completed") > 0,

                col("passed_tests") * 100.0 / col("tests_completed")

            ).otherwise(0.0)

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )